In [3]:
#由于kaggle来源的数据集已经去敏及数据预处理，下面清洗过程仅供参考(个人见解)
# 该notebook主要分为数据清洗或数据预处理(分别结合CDA2级教材4.3数据清洗和6.3数据预处理基础)
from glob import escape#导入必备的库
import numpy as np
import pandas as pd
# import toad(个人环境配置无法使用)
from sklearn.preprocessing import MinMaxScaler

#数据读取和统计值汇总
train_df=pd.read_csv(r"C:\Users\范彬\OneDrive\桌面\train.csv")#引用地址为个人习惯地址，可自行更改
train_df.head()
train_df.info()


# 重复值处理(此处基于CDA二级教程4.3.1重复值处理)
var = train_df[train_df.duplicated()]#查询重复值
if  var is None:
    escape()
else:
    train_df.drop_duplicates()#如果有重复值，去重
    print("The duplicate rows were dropped.")
# 缺失值处理(此基于CDA二级教程4.3.2缺失值处理)
esp=train_df.apply(lambda x:sum(x.isnull())/x.size)#查询缺失值
if esp is None:
    escape()
else:
    train_df.fillna(np.nan,inplace=True)
    print("The escaping contents were dropped.")

#离群值识别并处理
#这里结合CDA2级教程6.3.3中利用自定义函数使用盖帽法实现连续变量离群值识别和处理
# 类似方法如:四分位数法及分箱法，WoE法
def blk(floor,root):
    def f(x):
        if x < floor:
            x=floor
        elif x > root:
            x=root
        return x
    return f
q1=train_df["amt"].quantile(0.01)#计算百分位数
q99=train_df["amt"].quantile(0.99)#设置上下阀值
blk_tok=blk(q1,q99)
train_df["amt"]=train_df["amt"].map(blk_tok)#map用于series中对每个元素执行百分比操作
train_df["amt"].describe()

# 连续变量中心标准化或归一化(结合CDA2级教程6.3.7连续变量中心标准化，此处使用sklearn库实现极差标准化)
num_cols=["amt","city_pop"]# 选择特征列
scaler=MinMaxScaler()#初始归一化工具
train_df[num_cols]=scaler.fit_transform(train_df[num_cols])#拟合并转换
print(train_df[num_cols].head())

# Woe转换(结合CDA2级教程6.3.9Woe转换及相关资料查询所得)
# 1.使用toad库
# woe=toad.WOE()#初始化WOE工具
# woe.fit(train_df,y="is_fraud")#拟合WOE
# train_woe=woe.transform(train_df)#转换特征
# print(train_woe[["category","amt","gender"]].head())

# 2.手动计算Woe法
def calculate_woe(df,feature,target):
    #统计每个类别好/坏样本数量
    grouped=df.groupby(feature)[target].agg("sum","count")
    grouped["good"]=grouped["count"]-grouped["sum"]
    grouped["bad"]=grouped["sum"]
    #计算好坏样本数
    total_good=grouped["good"].sum()
    total_bad=grouped["bad"].sum()
    #计算Woe((Bi/BT)/(Gi/Gt))
    grouped["woe"]=np.log((grouped["bad"]/total_bad)/(grouped["bad"]/total_good))
    #映射回原数据
    woe_map=grouped["woe"].to_dict()
    df[f"{feature}_woe"]=df[feature].map(woe_map)
    return df
# 该Woe方法个人并未选择使用,代码也有一些瑕疵，个人环境出现标错，就不进行相关示例，个人大一学生能力有限，仅fanfare运行那不报错


















<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              